# O2 A optimal estimation

This notebook runs the same two-state aerosol retrieval in full mode and fastmode. The measurement is simulated from a truth case, then optimal estimation retrieves aerosol optical depth and aerosol layer mid-pressure.

In [ ]:
import statistics as stats
from time import perf_counter

from zdisamar import optimal_estimation as oe
from zdisamar import rtm
from zdisamar.rtm import SessionCache
from zdisamar.wavelength_bands import o2a

In [ ]:
def bench(label, fn, n=5):
    times = []
    for i in range(n):
        t0 = perf_counter()
        fn()
        dt = perf_counter() - t0
        times.append(dt)
        print(label, i, round(dt, 3))
    print(label, "min", round(min(times), 3), "median", round(stats.median(times), 3))

## Simulated truth and measurement

The truth case generates the synthetic reflectance measurement. Measurement noise is expressed as signal-to-noise ratio, either one scalar for the whole spectrum or one value per wavelength.

In [ ]:
truth = o2a.reference_scene()
truth.aerosol_optical_depth_550_nm = 0.18
truth.aerosol_layer.mid_pressure_hpa = 820.0

measurement = oe.simulate_measurement(truth, signal_to_noise=1000.0)
measurement.summary()

## Retrieval state

The state vector stays in physical coordinates. `prior_uncertainty` is the one-sigma prior spread in the same units as the corresponding state value.

In [ ]:
state_vector = oe.StateVector(
    (
        oe.AerosolOpticalDepth(
            initial=0.18,
            prior=0.18,
            prior_uncertainty=0.5,
            lower=0.0,
        ),
        oe.AerosolLayerMidPressure(
            initial=820.0,
            prior=820.0,
            prior_uncertainty=80.0,
        ),
    )
)
state_vector

## Full mode

Full mode uses the reference RTM settings for every OE step.

In [ ]:
full_case = o2a.reference_scene()

with SessionCache(full_case) as cache:
    full_result = oe.retrieve(
        scene=full_case,
        measurement=measurement,
        state_vector=state_vector,
        cache=cache,
    )

full_result.summary()

## Fastmode

Fastmode is enabled on the same case object. The retrieval call stays the same; the case-owned fastmode settings select the tuned RTM shortcuts, sparse fast-stage wavelengths, and sparse full-physics correction.

In [ ]:
fast_case = o2a.reference_scene()
fast_case.optimisation.fastmode.enabled = True
fast_case

In [ ]:
with SessionCache(fast_case) as cache:
    fast_result = oe.retrieve(
        scene=fast_case,
        measurement=measurement,
        state_vector=state_vector,
        cache=cache,
    )

fast_result.summary()

## Wall-clock timing

These calls include Python call overhead, case loading, native preparation, retrieval iterations, and the fastmode correction step.

In [ ]:
bench_full_case = o2a.reference_scene()
bench_fast_case = o2a.reference_scene()
bench_fast_case.optimisation.fastmode.enabled = True

bench(
    "full",
    lambda: oe.retrieve(
        scene=bench_full_case,
        measurement=measurement,
        state_vector=state_vector,
    ),
    n=3,
)

bench(
    "fast",
    lambda: oe.retrieve(
        scene=bench_fast_case,
        measurement=measurement,
        state_vector=state_vector,
    ),
    n=3,
)

## Tune fastmode knobs

The defaults are tuned for the retained O2 A validation sweep. For research, the same fields can be adjusted before running `oe.retrieve`.

In [ ]:
tuned_case = o2a.reference_scene()
tuned_case.optimisation.fastmode.enabled = True
fastmode = tuned_case.optimisation.fastmode

fastmode.radiative_transfer.fourier_order_cap = 5
fastmode.radiative_transfer.threshold_doubl = 3.0e-5
fastmode.adaptive_reference_grid.points_per_fwhm = 28

fastmode.oe.fast_stage_sampling.windows = (
    o2a.FastModeWavelengthWindow((755.0, 758.5), 16),
    o2a.FastModeWavelengthWindow((765.2, 768.0), 25),
)
fastmode.oe.final_correction.wavelength_count = 12

## Aerosol profile forward simulations

The forward model also accepts explicit aerosol profile layers for simulations that are not single-layer scalar cases.

In [ ]:
profile_case = o2a.reference_scene()
profile_case.aerosol_profile = (
    o2a.AerosolProfileLayer(
        top_pressure_hpa=620.0,
        bottom_pressure_hpa=700.0,
        optical_depth=0.10,
        single_scatter_albedo=0.94,
        asymmetry_factor=0.66,
    ),
    o2a.AerosolProfileLayer(
        top_pressure_hpa=700.0,
        bottom_pressure_hpa=820.0,
        optical_depth=0.22,
        single_scatter_albedo=0.92,
        asymmetry_factor=0.63,
    ),
)

profile_spectrum = rtm.spectrum(profile_case)
profile_spectrum.plot.reflectance()

## Retrieval plots

The package ships SVG plot accessors for the standard OE diagnostics.

In [ ]:
fast_result.plot.convergence()

In [ ]:
fast_result.plot.measurement_fit()

In [ ]:
fast_result.plot.jacobian()